In [2]:
import heapq
import time
import numpy as np
from scipy.sparse import load_npz
from scipy.spatial import cKDTree

# 1. Load đồ thị và chuyển sang dict kề
adj_matrix = load_npz("data/hcm_center_moto_adj.npz")
coords = np.load("data/hcm_center_moto_meta.npz")["coords"]
num_nodes = adj_matrix.shape[0]

graph = {i: [] for i in range(num_nodes)}
rows, cols = adj_matrix.nonzero()
for u, v, w in zip(rows, cols, adj_matrix.data):
    graph[int(u)].append((int(v), float(w)))

tree = cKDTree(coords)
people_gps = [
    (10.7725, 106.6980), # An
    (10.7827, 106.6959), # Bình
    (10.8013, 106.7114), # Chi
    (10.7995, 106.6922), # Phi
]
sources = [int(tree.query(gps)[1]) for gps in people_gps]
k = len(sources)

# 2. THUẬT TOÁN NƯỚC LOANG CHO MINIMAX (Dừng ngay khi gặp đỉnh đầu tiên đủ k người)
def wavefront_minimax(graph, sources, num_nodes):
    k = len(sources)
    T = np.full((k, num_nodes), np.inf, dtype=np.float64)
    visited_count = np.zeros(num_nodes, dtype=np.int32)
    
    pq = []
    for i, s in enumerate(sources):
        T[i, s] = 0.0
        heapq.heappush(pq, (0.0, i, s))
        
    pops_count = 0
    while pq:
        t, person_i, u = heapq.heappop(pq)
        if t > T[person_i, u]:
            continue
        pops_count += 1
        
        # Đánh dấu người person_i đã chính thức chốt xong thời gian tới đỉnh u
        visited_count[u] += 1
        if visited_count[u] == k:
            # Đỉnh đầu tiên được cả k người chạm tới -> TỐI ƯU TUYỆT ĐỐI!
            return u, t, pops_count, T
            
        for v, w in graph[u]:
            nt = t + w
            if nt < T[person_i, v]:
                T[person_i, v] = nt
                heapq.heappush(pq, (nt, person_i, v))
                
    return -1, np.inf, pops_count, T

# 3. THUẬT TOÁN NƯỚC LOANG CHO SUM (Branch & Bound với cận dưới k * t)
def wavefront_sum_exact(graph, sources, num_nodes):
    k = len(sources)
    T = np.full((k, num_nodes), np.inf, dtype=np.float64)
    visited_count = np.zeros(num_nodes, dtype=np.int32)
    partial_sum = np.zeros(num_nodes, dtype=np.float64)
    
    # Tập hợp các đỉnh đã có ít nhất 1 người nhưng chưa đủ k người chạm tới
    active_nodes = set()
    
    pq = []
    for i, s in enumerate(sources):
        T[i, s] = 0.0
        heapq.heappush(pq, (0.0, i, s))
        
    best_node = -1
    best_sum = np.inf
    pops_count = 0
    
    while pq:
        t, person_i, u = heapq.heappop(pq)
        if t > T[person_i, u]:
            continue
            
        # Tính cận dưới toàn cục chính xác (Exact Global Lower Bound)
        if best_sum < np.inf:
            # Cận dưới của các đỉnh chưa ai chạm tới là k * t
            global_lb = k * t
            # Kiểm tra cận dưới của các đỉnh đang dang dở trong active_nodes
            for v_act in active_nodes:
                lb_v = partial_sum[v_act] + (k - visited_count[v_act]) * t
                if lb_v < global_lb:
                    global_lb = lb_v
                    
            if global_lb >= best_sum:
                break
                
        pops_count += 1
        visited_count[u] += 1
        partial_sum[u] += t
        
        if visited_count[u] == 1:
            active_nodes.add(u)
        elif visited_count[u] == k:
            active_nodes.discard(u)
            if partial_sum[u] < best_sum:
                best_sum = partial_sum[u]
                best_node = u
                
        for v, w in graph[u]:
            nt = t + w
            if nt < T[person_i, v]:
                T[person_i, v] = nt
                heapq.heappush(pq, (nt, person_i, v))
                
    return best_node, best_sum, pops_count, T

# Chạy kiểm chứng lại Wavefront Sum chuẩn xác
opt_sum_node, opt_sum_sec, pops_sum, _ = wavefront_sum_exact(graph, sources, num_nodes)
print(f"[Wavefront Sum Exact] Đỉnh tối ưu: #{opt_sum_node} | Tổng = {opt_sum_sec/60:.2f} phút")
print(f"  -> Số lượt mở rộng đỉnh: {pops_sum:,} / {k * num_nodes:,} (Chỉ quét {pops_sum/(k*num_nodes)*100:.1f}% không gian!)")



[Wavefront Sum Exact] Đỉnh tối ưu: #1828 | Tổng = 28.16 phút
  -> Số lượt mở rộng đỉnh: 24,662 / 56,416 (Chỉ quét 43.7% không gian!)
